# Exercises XP - Heart Disease Prediction

## What you will learn
- Load and inspect CSV data
- EDA and preprocessing
- Train Logistic Regression, SVM, XGBoost
- Hyperparameter tuning with GridSearchCV
- Evaluate with standard metrics

## What you will create
- Working classifiers and a simple comparison report


## Setup

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
except Exception:
    # !pip install xgboost
    XGBClassifier = None

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Exercise 1 - Exploratory Data Analysis

In [ ]:
# ----- Load CSV -----
# Update CSV_PATH to your local path or keep the default for Colab
CSV_PATH = 'dataset_heart.csv'  # adjust if needed
df = pd.read_csv(CSV_PATH)

print('Shape:', df.shape)
print()
df.head()

In [ ]:
# Basic inspection
print(df.info())
print()
print(df.describe())

In [ ]:
# The dataset uses 'heart disease' as the target column.
# Values: 1 = no disease, 2 = disease  -> convert to binary 0/1
target_col = 'heart disease'
print('Target value counts:')
print(df[target_col].value_counts())

# Binarise: 2 -> 1 (disease present), 1 -> 0 (no disease)
df['target'] = (df[target_col] == 2).astype(int)

# Features and target
X = df.drop(columns=[target_col, 'target'])
y = df['target']

print('\nBinary target counts:')
print(y.value_counts())

In [ ]:
# Train / test split with stratification to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train size:', X_train.shape)
print('Test  size:', X_test.shape)

### Basic visual checks

In [ ]:
# Histograms of a few numeric features
numeric_preview = ['age', 'resting blood pressure', 'serum cholestoral', 'max heart rate']
df[numeric_preview].hist(bins=20, figsize=(12, 8), color='steelblue', edgecolor='white')
plt.suptitle('Distribution of key numeric features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Class balance
ax = y.value_counts().sort_index().plot(kind='bar', color=['#4c72b0', '#dd8452'],
                                         edgecolor='white', figsize=(5, 4))
ax.set_xticklabels(['No disease (0)', 'Disease (1)'], rotation=0)
ax.set_title('Class balance')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(f'Class ratio  0:{y.value_counts()[0]}  1:{y.value_counts()[1]}')

## Preprocessing pipeline

In [ ]:
# All columns in this dataset are numeric, so no one-hot encoding needed.
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_cols = X.select_dtypes(include='number').columns.tolist()

print('Categorical columns:', cat_cols)
print('Numeric columns    :', num_cols)

# ColumnTransformer: scale numeric features
pre = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols)
    ],
    remainder='drop'
)

## Helper - evaluation function

In [ ]:
def eval_and_report(name, model, X_test, y_test):
    """Compute metrics, print them, draw confusion matrix and ROC curve."""
    # Predictions
    y_pred = model.predict(X_test)

    # Core metrics
    metrics = {
        'accuracy' : accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall'   : recall_score(y_test, y_pred),
        'f1'       : f1_score(y_test, y_pred),
    }

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # Confusion matrix
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred),
        display_labels=['No disease', 'Disease']
    ).plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title(f'{name} - Confusion Matrix')

    # ROC curve (only if model supports predict_proba)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        metrics['roc_auc'] = auc
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        axes[1].plot(fpr, tpr, label=f'AUC = {auc:.3f}')
        axes[1].plot([0, 1], [0, 1], 'k--')
        axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
        axes[1].set_title(f'{name} - ROC Curve')
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, 'ROC not available\n(no predict_proba)',
                     ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title(f'{name} - ROC Curve')

    plt.tight_layout()
    plt.show()

    print(name, {k: round(v, 4) for k, v in metrics.items()})
    return metrics

## Exercise 2 - Logistic Regression without Grid Search

In [ ]:
# Pipeline: preprocessing + Logistic Regression
pipe_lr = Pipeline([
    ('pre', pre),
    ('lr', LogisticRegression(
        solver='liblinear',   # works well for small datasets
        C=1.0,                # inverse of regularisation strength
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

pipe_lr.fit(X_train, y_train)
lr_no_gs_metrics = eval_and_report('LR (no grid search)', pipe_lr, X_test, y_test)

## Exercise 3 - Logistic Regression with Grid Search

In [ ]:
pipe_lr_cv = Pipeline([
    ('pre', pre),
    ('lr', LogisticRegression(
        solver='liblinear',
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

# Search over regularisation strength and penalty type
# liblinear supports L1 (sparse) and L2 (ridge) penalties
param_grid = {
    'lr__C'      : [0.01, 0.1, 1, 10, 100],
    'lr__penalty': ['l1', 'l2'],
}

grid_lr = GridSearchCV(
    pipe_lr_cv,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_lr.fit(X_train, y_train)

print('Best params:', grid_lr.best_params_)
print('Best CV F1 :', round(grid_lr.best_score_, 4))

best_lr = grid_lr.best_estimator_
lr_gs_metrics = eval_and_report('LR (grid search)', best_lr, X_test, y_test)

## Exercise 4 - SVM without Grid Search

In [ ]:
# RBF kernel is a good default: handles non-linear boundaries
# C=1.0 and gamma='scale' are scikit-learn defaults
pipe_svm = Pipeline([
    ('pre', pre),
    ('svm', SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        random_state=RANDOM_STATE
    ))
])

pipe_svm.fit(X_train, y_train)
svm_no_metrics = eval_and_report('SVM (no grid search)', pipe_svm, X_test, y_test)

## Exercise 5 - SVM with Grid Search

In [ ]:
# probability=True enables predict_proba for the ROC curve
pipe_svm_cv = Pipeline([
    ('pre', pre),
    ('svm', SVC(probability=True, random_state=RANDOM_STATE))
])

svm_param_grid = {
    'svm__kernel': ['rbf', 'linear'],
    'svm__C'     : [0.1, 1, 10],
    'svm__gamma' : ['scale', 'auto'],  # ignored for linear kernel but harmless
}

grid_svm = GridSearchCV(
    pipe_svm_cv,
    svm_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_svm.fit(X_train, y_train)

print('Best params:', grid_svm.best_params_)
print('Best CV F1 :', round(grid_svm.best_score_, 4))

best_svm = grid_svm.best_estimator_
svm_gs_metrics = eval_and_report('SVM (grid search)', best_svm, X_test, y_test)

## Exercise 6 - XGBoost without Grid Search

In [ ]:
# Choices justified:
#   n_estimators=300 : enough trees to learn patterns without being slow
#   learning_rate=0.1: standard starting point (balances speed vs accuracy)
#   max_depth=4      : moderate depth prevents overfitting on this small dataset
pipe_xgb = Pipeline([
    ('pre', pre),
    ('xgb', XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=4,
        random_state=RANDOM_STATE,
        eval_metric='logloss'
    ))
])

pipe_xgb.fit(X_train, y_train)
xgb_no_metrics = eval_and_report('XGBoost (no grid search)', pipe_xgb, X_test, y_test)

## Exercise 7 - XGBoost with Grid Search

In [ ]:
pipe_xgb_cv = Pipeline([
    ('pre', pre),
    ('xgb', XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric='logloss'
    ))
])

xgb_param_grid = {
    'xgb__n_estimators' : [100, 300],
    'xgb__learning_rate': [0.05, 0.1],
    'xgb__max_depth'    : [3, 5],
    'xgb__subsample'    : [0.8, 1.0],
}

grid_xgb = GridSearchCV(
    pipe_xgb_cv,
    xgb_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_xgb.fit(X_train, y_train)

print('Best params:', grid_xgb.best_params_)
print('Best CV F1 :', round(grid_xgb.best_score_, 4))

best_xgb = grid_xgb.best_estimator_
xgb_gs_metrics = eval_and_report('XGBoost (grid search)', best_xgb, X_test, y_test)

## Compare models

In [ ]:
# Collect all metrics into a single comparison table
summary = {
    'LR (no grid)'   : lr_no_gs_metrics,
    'LR (grid)'      : lr_gs_metrics,
    'SVM (no grid)'  : svm_no_metrics,
    'SVM (grid)'     : svm_gs_metrics,
    'XGB (no grid)'  : xgb_no_metrics,
    'XGB (grid)'     : xgb_gs_metrics,
}

summary_df = pd.DataFrame(summary).T.round(4)
print(summary_df.to_string())
summary_df

In [ ]:
# Bar chart comparison of F1 and accuracy across all models
plot_cols = ['accuracy', 'f1']
ax = summary_df[plot_cols].plot(
    kind='bar', figsize=(10, 5), color=['#4c72b0', '#dd8452'],
    edgecolor='white', width=0.7
)
ax.set_ylim(0.6, 1.0)
ax.set_ylabel('Score')
ax.set_title('Model Comparison - Accuracy & F1')
ax.legend(loc='lower right')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()